In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Cleeveland_Dataset_no_thal.csv to Cleeveland_Dataset_no_thal.csv


In [3]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print("Initial shape:", df.shape)
df.head()

Initial shape: (303, 13)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,target
0,63,1,0,145,233,1,2,150,0,2.3,2,0,0
1,67,1,3,160,286,0,2,108,1,1.5,1,3,1
2,67,1,3,120,229,0,2,129,1,2.6,1,2,1
3,37,1,2,130,250,0,0,187,0,3.5,2,0,0
4,41,0,1,130,204,0,2,172,0,1.4,0,0,0


In [4]:
print("Data types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:\n", df['target'].value_counts())

Data types:
 age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
target        int64
dtype: object

Missing values per column:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
target      0
dtype: int64

Duplicate rows: 0

Target distribution:
 target
0    164
1    139
Name: count, dtype: int64


In [5]:
df = df.drop_duplicates()
df = df.dropna()
print("Shape after cleaning:", df.shape)
print("Duplicate rows remaining:", df.duplicated().sum())
print("Missing values remaining:", df.isnull().sum().sum())

Shape after cleaning: (303, 13)
Duplicate rows remaining: 0
Missing values remaining: 0


In [6]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTraining target distribution:\n", y_train.value_counts(normalize=True))
print("\nTest target distribution:\n", y_test.value_counts(normalize=True))

Training set shape: (242, 12)
Test set shape: (61, 12)

Training target distribution:
 target
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Test target distribution:
 target
0    0.540984
1    0.459016
Name: proportion, dtype: float64


In [7]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Missing values in X_train after imputation:", X_train_imputed.isnull().sum().sum())
print("Missing values in X_test after imputation:", X_test_imputed.isnull().sum().sum())
print("\nX_train_imputed shape:", X_train_imputed.shape)
print("X_test_imputed shape:", X_test_imputed.shape)

Missing values in X_train after imputation: 0
Missing values in X_test after imputation: 0

X_train_imputed shape: (242, 12)
X_test_imputed shape: (61, 12)


In [8]:
def get_iqr_bounds(train_col):
    Q1 = train_col.quantile(0.25)
    Q3 = train_col.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

numeric_cols = X_train_imputed.columns.tolist()
bounds = {}

for col in numeric_cols:
    lower, upper = get_iqr_bounds(X_train_imputed[col])
    bounds[col] = (lower, upper)
    n_outliers_train = ((X_train_imputed[col] < lower) | (X_train_imputed[col] > upper)).sum()
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}, outliers_in_train={n_outliers_train}")

age: lower=28.50, upper=80.50, outliers_in_train=0
sex: lower=-1.50, upper=2.50, outliers_in_train=0
cp: lower=-1.38, upper=5.62, outliers_in_train=0
trestbps: lower=90.00, upper=170.00, outliers_in_train=6
chol: lower=113.38, upper=376.38, outliers_in_train=5
fbs: lower=0.00, upper=0.00, outliers_in_train=35
restecg: lower=-3.00, upper=5.00, outliers_in_train=0
thalach: lower=87.25, upper=213.25, outliers_in_train=1
exang: lower=-1.50, upper=2.50, outliers_in_train=0
oldpeak: lower=-2.40, upper=4.00, outliers_in_train=4
slope: lower=-1.50, upper=2.50, outliers_in_train=0
ca: lower=-1.50, upper=2.50, outliers_in_train=12


In [9]:
continuous_cols = ['trestbps', 'chol', 'thalach', 'oldpeak']

X_train_capped = X_train_imputed.copy()
X_test_capped = X_test_imputed.copy()

for col in continuous_cols:
    lower, upper = bounds[col]
    X_train_capped[col] = X_train_capped[col].clip(lower, upper)
    X_test_capped[col] = X_test_capped[col].clip(lower, upper)

for col in continuous_cols:
    lower, upper = bounds[col]
    n_train = ((X_train_capped[col] < lower) | (X_train_capped[col] > upper)).sum()
    n_test = ((X_test_capped[col] < lower) | (X_test_capped[col] > upper)).sum()
    print(f"{col}: outliers_in_train={n_train}, outliers_in_test={n_test}")

trestbps: outliers_in_train=0, outliers_in_test=0
chol: outliers_in_train=0, outliers_in_test=0
thalach: outliers_in_train=0, outliers_in_test=0
oldpeak: outliers_in_train=0, outliers_in_test=0


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_capped),
    columns=X_train_capped.columns,
    index=X_train_capped.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_capped),
    columns=X_test_capped.columns,
    index=X_test_capped.index
)

print("X_train_scaled stats:\n", X_train_scaled.describe().loc[['mean', 'std']])
print("\nX_test_scaled shape:", X_test_scaled.shape)
X_train_scaled.head()

X_train_scaled stats:
                age           sex            cp      trestbps          chol  \
mean -1.835079e-16  1.027644e-16  1.807553e-16  8.074349e-16 -1.908483e-16   
std   1.002073e+00  1.002073e+00  1.002073e+00  1.002073e+00  1.002073e+00   

               fbs       restecg       thalach         exang       oldpeak  \
mean -2.202095e-17  5.964008e-17 -1.468064e-16  1.376310e-18  1.468064e-17   
std   1.002073e+00  1.002073e+00  1.002073e+00  1.002073e+00  1.002073e+00   

             slope            ca  
mean -8.074349e-17  7.340318e-17  
std   1.002073e+00  1.002073e+00  

X_test_scaled shape: (61, 12)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca
180,-0.729485,0.68313,0.870169,-0.398560,0.531065,-0.411196,1.022996,0.712047,-0.696177,-0.455191,0.675060,-0.689715
208,0.050166,0.68313,-1.184278,-0.037318,0.280548,-0.411196,-0.981579,0.221597,-0.696177,-0.927560,-0.958585,-0.689715
167,-0.061212,-1.46385,-1.184278,0.083095,0.823334,2.431930,1.022996,0.399942,1.436416,-0.927560,-0.958585,0.445734
105,-0.061212,0.68313,-1.184278,-1.361870,1.261738,-0.411196,-0.981579,0.266183,-0.696177,-0.927560,-0.958585,-0.689715
297,0.272924,-1.46385,0.870169,0.564751,-0.157856,-0.411196,-0.981579,-1.205170,1.436416,-0.738613,0.675060,-0.689715


In [11]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_scaled, y_train), 1):
    fold_train_dist = y_train.iloc[train_idx].value_counts(normalize=True).to_dict()
    fold_val_dist = y_train.iloc[val_idx].value_counts(normalize=True).to_dict()
    print(f"Fold {fold}: train_size={len(train_idx)}, val_size={len(val_idx)}, "
          f"val_class_dist={fold_val_dist}")

Fold 1: train_size=193, val_size=49, val_class_dist={0: 0.5306122448979592, 1: 0.46938775510204084}
Fold 2: train_size=193, val_size=49, val_class_dist={0: 0.5510204081632653, 1: 0.4489795918367347}
Fold 3: train_size=194, val_size=48, val_class_dist={0: 0.5416666666666666, 1: 0.4583333333333333}
Fold 4: train_size=194, val_size=48, val_class_dist={0: 0.5416666666666666, 1: 0.4583333333333333}
Fold 5: train_size=194, val_size=48, val_class_dist={0: 0.5416666666666666, 1: 0.4583333333333333}


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [19]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
print("Logistic Regression trained.")

Logistic Regression trained.


In [20]:
rf_model = RandomForestClassifier(random_state=42, n_estimators=200)
rf_model.fit(X_train_scaled, y_train)
print("Random Forest trained.")

Random Forest trained.


In [21]:
rf_model = RandomForestClassifier(random_state=42, n_estimators=200)
rf_model.fit(X_train_scaled, y_train)
print("Random Forest trained.")

Random Forest trained.


In [22]:
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)
print("XGBoost trained.")

XGBoost trained.


In [23]:
cat_model = CatBoostClassifier(random_state=42, verbose=0)
cat_model.fit(X_train_scaled, y_train)
print("CatBoost trained.")

CatBoost trained.


In [24]:
from sklearn.model_selection import cross_val_score

lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"Logistic Regression: fold_scores={np.round(lr_cv_scores, 4)}, mean={lr_cv_scores.mean():.4f}, std={lr_cv_scores.std():.4f}")

Logistic Regression: fold_scores=[0.7755 0.8163 0.7292 0.8542 0.8542], mean=0.8059, std=0.0481


In [25]:
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"Random Forest: fold_scores={np.round(rf_cv_scores, 4)}, mean={rf_cv_scores.mean():.4f}, std={rf_cv_scores.std():.4f}")

Random Forest: fold_scores=[0.7755 0.7959 0.75   0.9167 0.6667], mean=0.7810, std=0.0809


In [26]:
xgb_cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"XGBoost: fold_scores={np.round(xgb_cv_scores, 4)}, mean={xgb_cv_scores.mean():.4f}, std={xgb_cv_scores.std():.4f}")

XGBoost: fold_scores=[0.7755 0.7551 0.7292 0.8125 0.7083], mean=0.7561, std=0.0362


In [27]:
cat_cv_scores = cross_val_score(cat_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"CatBoost: fold_scores={np.round(cat_cv_scores, 4)}, mean={cat_cv_scores.mean():.4f}, std={cat_cv_scores.std():.4f}")

CatBoost: fold_scores=[0.8163 0.7755 0.7708 0.8542 0.625 ], mean=0.7684, std=0.0778


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_results = {
    'Accuracy': accuracy_score(y_test, lr_pred),
    'Precision': precision_score(y_test, lr_pred),
    'Recall': recall_score(y_test, lr_pred),
    'F1-Score': f1_score(y_test, lr_pred),
    'ROC-AUC': roc_auc_score(y_test, lr_proba)
}

print("Logistic Regression:")
for metric, value in lr_results.items():
    print(f"  {metric}: {value:.4f}")

Logistic Regression:
  Accuracy: 0.9180
  Precision: 0.8710
  Recall: 0.9643
  F1-Score: 0.9153
  ROC-AUC: 0.9578


In [29]:
rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_results = {
    'Accuracy': accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall': recall_score(y_test, rf_pred),
    'F1-Score': f1_score(y_test, rf_pred),
    'ROC-AUC': roc_auc_score(y_test, rf_proba)
}

print("Random Forest:")
for metric, value in rf_results.items():
    print(f"  {metric}: {value:.4f}")

Random Forest:
  Accuracy: 0.9344
  Precision: 0.9000
  Recall: 0.9643
  F1-Score: 0.9310
  ROC-AUC: 0.9724


In [30]:
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_results = {
    'Accuracy': accuracy_score(y_test, xgb_pred),
    'Precision': precision_score(y_test, xgb_pred),
    'Recall': recall_score(y_test, xgb_pred),
    'F1-Score': f1_score(y_test, xgb_pred),
    'ROC-AUC': roc_auc_score(y_test, xgb_proba)
}

print("XGBoost:")
for metric, value in xgb_results.items():
    print(f"  {metric}: {value:.4f}")

XGBoost:
  Accuracy: 0.8525
  Precision: 0.7879
  Recall: 0.9286
  F1-Score: 0.8525
  ROC-AUC: 0.9535


In [31]:
cat_pred = cat_model.predict(X_test_scaled)
cat_proba = cat_model.predict_proba(X_test_scaled)[:, 1]

cat_results = {
    'Accuracy': accuracy_score(y_test, cat_pred),
    'Precision': precision_score(y_test, cat_pred),
    'Recall': recall_score(y_test, cat_pred),
    'F1-Score': f1_score(y_test, cat_pred),
    'ROC-AUC': roc_auc_score(y_test, cat_proba)
}

print("CatBoost:")
for metric, value in cat_results.items():
    print(f"  {metric}: {value:.4f}")

CatBoost:
  Accuracy: 0.9344
  Precision: 0.9000
  Recall: 0.9643
  F1-Score: 0.9310
  ROC-AUC: 0.9686


In [32]:
comparison_df = pd.DataFrame({
    'Logistic Regression': lr_results,
    'Random Forest': rf_results,
    'XGBoost': xgb_results,
    'CatBoost': cat_results
}).T

comparison_df = comparison_df.round(4)
comparison_df = comparison_df.sort_values(by='F1-Score', ascending=False)

print("Comparative Analysis Table:\n")
comparison_df

Comparative Analysis Table:



,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Random Forest,0.9344,0.9000,0.9643,0.9310,0.9724
CatBoost,0.9344,0.9000,0.9643,0.9310,0.9686
Logistic Regression,0.9180,0.8710,0.9643,0.9153,0.9578
XGBoost,0.8525,0.7879,0.9286,0.8525,0.9535
